In [6]:
import pandas as pd
import os
from tqdm.auto import tqdm

# Base directory
str_oecd_base_dir = '/Users/xingyuanzhao/Documents/projects/ai_policy/data/OECD'

# Get list of directories
list_subdirs = [d for d in os.listdir(str_oecd_base_dir) if os.path.isdir(os.path.join(str_oecd_base_dir, d))]

# Iterate through subfolders in OECD directory
dict_dataframes = {}

with tqdm(total=len(list_subdirs), position=0, leave=True) as pbar:
    for str_item in list_subdirs:
        str_item_path = os.path.join(str_oecd_base_dir, str_item)
        
        # Find CSV files in this folder
        list_csv_files = [f for f in os.listdir(str_item_path) if f.endswith('.csv')]
        
        if list_csv_files:
            for str_csv_file in list_csv_files:
                str_csv_path = os.path.join(str_item_path, str_csv_file)
                
                # Create variable name from folder name
                str_var_name = 'df_' + str_item.replace(' ', '_').replace(',', '').lower()
                
                # Read CSV
                df_current = pd.read_csv(str_csv_path)
                dict_dataframes[str_var_name] = df_current
        
        pbar.update(1)

print("\n=== All data loaded successfully ===")
print(f"Total dataframes: {len(dict_dataframes)}")
print(f"Dataframe variables: {list(dict_dataframes.keys())}")


100%|██████████| 7/7 [00:00<00:00, 1134.16it/s]


=== All data loaded successfully ===
Total dataframes: 7
Dataframe variables: ['df_n_models_on_hugging_face_by_host', 'df_n_models_on_hugging_face_by_prov', 'df_talent_flow', 'df_n_models_on_hugging_face_by_prov_foundation', 'df_n_models_on_hugging_face_by_dev', 'df_q_models_on_hugging_face', 'df_vc_invest_in_ai']


In [7]:
# Iterate through all dataframes
with tqdm(total=len(dict_dataframes), position=0, leave=True) as pbar:
    for str_df_name, df_current in dict_dataframes.items():
        # Get column names
        list_col_names = df_current.columns.tolist()
        
        # Check each column
        for str_col_name in list_col_names:
            # If column name has ref_date
            if 'ref_date' in str_col_name.lower():
                # Convert YYYYMM to YYYY/MM/DD with day as 01
                df_current[str_col_name] = pd.to_datetime(df_current[str_col_name].astype(str), format='%Y%m')
                df_current[str_col_name] = df_current[str_col_name].dt.date
            
            # If column name has year
            elif 'year' in str_col_name.lower():
                # Convert YYYY to YYYY/MM/DD with month and day as 01
                df_current[str_col_name] = pd.to_datetime(df_current[str_col_name].astype(str) + '0101', format='%Y%m%d')
                df_current[str_col_name] = df_current[str_col_name].dt.date
        
        # Update the dataframe in dictionary
        dict_dataframes[str_df_name] = df_current
        
        pbar.update(1)

print("=== Date conversion completed ===")


100%|██████████| 7/7 [00:00<00:00, 379.30it/s]

=== Date conversion completed ===


In [12]:
# Create output directory if not exists
str_output_dir = '/Users/xingyuanzhao/Documents/projects/ai_policy/data/OECD_cleaned'
os.makedirs(str_output_dir, exist_ok=True)

# Save all dataframes
with tqdm(total=len(dict_dataframes), position=0, leave=True) as pbar:
    for str_df_name, df_current in dict_dataframes.items():
        # Create file path with dict key as filename
        str_file_path = os.path.join(str_output_dir, f'{str_df_name}.csv')
        
        # Save dataframe to CSV
        df_current.to_csv(str_file_path, index=False)
        
        pbar.update(1)

print(f"=== All dataframes saved to {str_output_dir} ===")


100%|██████████| 7/7 [00:00<00:00, 811.61it/s]

=== All dataframes saved to /Users/xingyuanzhao/Documents/projects/ai_policy/data/OECD_cleaned ===


In [15]:
# Read benchmark_paper_stock_trade_cov.csv
str_file_path = '/Users/xingyuanzhao/Documents/projects/ai_policy/data/hugging_face/benchmark_paper_stock_trade_cov.csv'
df_benchmark_stock_trade_cov = pd.read_csv(str_file_path)

# Convert year_month column
if 'year_month' in df_benchmark_stock_trade_cov.columns:
    # Convert YYYYMM to YYYY/MM/DD with day as 01
    df_benchmark_stock_trade_cov['year_month'] = pd.to_datetime(df_benchmark_stock_trade_cov['year_month'].astype(str), format='%Y%m')
    df_benchmark_stock_trade_cov['year_month'] = df_benchmark_stock_trade_cov['year_month'].dt.date

# Save to hugging_face folder
df_benchmark_stock_trade_cov.to_csv('/Users/xingyuanzhao/Documents/projects/ai_policy/data/hugging_face/leader_board_with_cov_cleaned.csv', index=False)

print("=== Benchmark stock trade data loaded and date converted ===")


=== Benchmark stock trade data loaded and date converted ===
